# Unstructured 2D Mesh: Diffusion in a Triangular Mesh

This notebook demonstrates how to use PyFVTool's new unstructured mesh support for solving diffusion problems on triangular meshes.

## Overview

PyFVTool now supports unstructured meshes through a unified connectivity-based architecture. This example shows:

1. Creating an unstructured triangular mesh using Delaunay triangulation
2. Tagging boundaries by geometric location
3. Setting boundary conditions using tag names
4. Solving a steady-state diffusion equation
5. Visualizing the solution on the triangular mesh

## Problem Setup

We solve Laplace's equation $\nabla^2 \phi = 0$ on a rectangular domain $[0,1] \times [0,0.5]$ with boundary conditions:
- Left ($x=0$): $\phi = 0$ (Dirichlet)
- Right ($x=1$): $\phi = 1$ (Dirichlet)
- Top and bottom: $\partial \phi / \partial n = 0$ (Neumann, zero flux)

The analytical solution is $\phi(x,y) = x$ (linear variation in x-direction).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyfvtool as pf

print('PyFVTool version:', pf.__version__)

## Step 1: Create an Unstructured Triangular Mesh

We use `UnstructuredMesh2D.from_delaunay()` to create a triangular mesh of a rectangular domain. This method uses `scipy.spatial.Delaunay` internally.

Parameters:
- `Nx=20`, `Ny=10`: Approximate number of cells in x and y directions
- `Lx=1.0`, `Ly=0.5`: Domain dimensions

In [ ]:
# Create a triangular mesh of a rectangular domain using Delaunay
print('Creating unstructured 2D mesh...')
mesh = pf.UnstructuredMesh2D.from_delaunay(20, 10, 1.0, 0.5)
print(f'Mesh created: {mesh.num_cells} cells, {mesh.num_faces} faces')

## Step 2: Tag Boundaries by Geometric Location

For rectangular domains, we can automatically identify boundary faces by their geometric location using `UnstructuredMesh2D.geometric_boundary_tags()`. This function returns a dictionary mapping tag names ('left', 'right', 'bottom', 'top') to face indices.

We then create a new mesh with these boundary tags. This overwrites the default 'boundary' tag.

In [ ]:
# Use geometric boundary tagging to identify left/right/top/bottom
print('Tagging boundaries by geometric location...')
boundary_tags = pf.UnstructuredMesh2D.geometric_boundary_tags(
    mesh, x_range=(0.0, 1.0), y_range=(0.0, 0.5)
)

# Re-create mesh with these boundary tags (overwrites default "boundary")
mesh2 = pf.UnstructuredMesh2D(mesh._nodes, mesh._cells, boundary_tags)

print(f"Boundary tags: {list(boundary_tags.keys())}")
for tag, faces in boundary_tags.items():
    print(f'  {tag}: {len(faces)} faces')

## Step 3: Set Boundary Conditions

We create a `BoundaryConditions` object and assign conditions to each tag group. The syntax `BC.left`, `BC.right`, etc. works because the mesh has tags with those names.

Boundary conditions follow the Robin formula: $a \nabla \phi \cdot \vec{n} + b \phi = c$
- Dirichlet: $a = 0$, $b = 1$, $c = \phi_{boundary}$
- Neumann (zero flux): $a = 1$, $b = 0$, $c = 0$ (default)

In [ ]:
# Set up boundary conditions
print('Setting boundary conditions...')
BC = pf.BoundaryConditions(mesh2)

# Left: Dirichlet phi = 0
BC.left.a[:] = 0.0
BC.left.b[:] = 1.0
BC.left.c[:] = 0.0

# Right: Dirichlet phi = 1  
BC.right.a[:] = 0.0
BC.right.b[:] = 1.0
BC.right.c[:] = 1.0

# Top and bottom: zero flux (default Neumann a=1, b=0, c=0)
print('  Left: Dirichlet phi=0')
print('  Right: Dirichlet phi=1')
print('  Top/bottom: Neumann zero flux (default)')

## Step 4: Create Cell Variable and Diffusion Coefficient

We create a `CellVariable` for the solution $\phi$ with the boundary conditions attached. The diffusion coefficient is also a `CellVariable` (constant = 1.0), which we average to faces using harmonic mean.

In [ ]:
# Create cell variable with initial value
phi = pf.CellVariable(mesh2, 0.0, BC)

# Set diffusion coefficient
D = pf.CellVariable(mesh2, 1.0)
D_face = pf.harmonicMean(D)

print(f'Created cell variable phi with {len(phi.value)} interior cells')

## Step 5: Build and Solve the Linear System

We build the diffusion matrix using `diffusionTerm()` and add boundary conditions using `boundaryConditionsTerm()`. The combined system $M \phi = RHS$ is solved with `solveMatrixPDE()`.

Note: The unified connectivity-based discretization works the same for structured and unstructured meshes!

In [ ]:
# Build and solve the linear system
print('Building diffusion term...')
Mdiff = pf.diffusionTerm(D_face)
Mbc, RHSbc = pf.boundaryConditionsTerm(BC)

M = Mdiff + Mbc
RHS = RHSbc

print('Solving linear system...')
phi_sol = pf.solveMatrixPDE(mesh2, M, RHS)

print('Solution computed successfully!')

## Step 6: Visualize the Solution

We visualize both the mesh and the solution using Matplotlib's triangular plotting functions:
- `triplot()` shows the mesh structure
- `tripcolor()` shows the cell-centered solution with flat shading

The solution should show a linear gradient from left (phi=0) to right (phi=1).

In [ ]:
# Visualize the solution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Mesh triangulation
ax1.set_aspect('equal')
ax1.triplot(mesh2._nodes[:, 0], mesh2._nodes[:, 1], mesh2._cells)
ax1.set_title('Triangular Mesh')
ax1.set_xlabel('x')
ax1.set_ylabel('y')

# Plot 2: Solution
ax2.set_aspect('equal')
tcf = ax2.tripcolor(mesh2._nodes[:, 0], mesh2._nodes[:, 1], 
                    mesh2._cells, phi_sol.value, shading='flat', cmap='viridis')
ax2.set_title(r'Solution $\phi(x,y)$')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(tcf, ax=ax2, label=r'$\phi$')

plt.tight_layout()
plt.show()

## Step 7: Analyze Results

We compute some statistics and compare with the analytical solution $\phi(x) = x$. Due to mesh discretization error, the solution won't be exactly linear, but should be close.

In [ ]:
# Compute and print some statistics
print('\nSolution statistics:')
print(f'  Min: {phi_sol.value.min():.4f}')
print(f'  Max: {phi_sol.value.max():.4f}')
print(f'  Mean: {phi_sol.value.mean():.4f}')

# Expected linear variation in x-direction
x_cells = mesh2._cell_centers[:, 0]
phi_expected = x_cells  # phi(x) = x for pure diffusion with these BCs
error = np.abs(phi_sol.value - phi_expected).max()
print(f'  Max error from linear solution phi(x)=x: {error:.4f}')
print(f'  Relative error: {error:.2%}')

## Summary

This example demonstrates the key steps for using unstructured meshes in PyFVTool:

1. **Mesh creation**: Use `UnstructuredMesh2D.from_delaunay()` or construct from nodes/cells arrays
2. **Boundary tagging**: Use geometric tagging or custom tags
3. **BC specification**: Assign Robin coefficients to tag groups
4. **Term assembly**: Use the same `diffusionTerm()`, `convectionTerm()`, etc. functions
5. **Solving**: Use `solveMatrixPDE()` or `solvePDE()`
6. **Visualization**: Use Matplotlib's triangular plotting functions

The same workflow applies to 3D tetrahedral meshes using `UnstructuredMesh3D`.